# 00 — Acesso aos dados (Parquet)

Valida o acesso ao dataset distribuído via GitHub Release.

Pré-requisitos:
- Estar na raiz do projeto (ou em `notebooks/`).
- `requirements.txt` já instalado.
- Para repositório privado: variável/segregado `GITHUB_TOKEN` configurado.

## 0. Configuração inicial (imports e `sys.path`)

Deixe **todos os imports** aqui nesta primeira célula. Isso evita avisos do Pylance nas células seguintes (o analisador estático não acompanha a ordem de execução do kernel).

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import storage

print("ROOT_DIR:", ROOT_DIR)

# 1. Diagnóstico rápido (rode isto antes de qualquer coisa)

In [ ]:
path = storage.get_parquet(force=True)

print("Arquivo:", path)
print("Existe:", path.exists())
print("Tamanho:", path.stat().st_size, "bytes")

with open(path, "rb") as f:
    print("Primeiros 20 bytes:", f.read(20))

In [ ]:
import pyarrow.parquet as pq
pq.read_schema(path)

## 2. Leitura mínima (só uma coluna)

Verifica se o Parquet é baixado/aberto corretamente.

In [ ]:
df = storage.read_srag(columns=["sg_uf"])
print("shape:", df.shape)
df.head()

## 3. Coluna categórica — municípios únicos

In [ ]:
storage.read_srag(columns=["id_municip"]).drop_duplicates().head(20)

## 4. Leitura com filtro (pushdown do Parquet)

Só as colunas necessárias e apenas linhas de SP.

In [ ]:
df_sp = storage.read_srag(
    columns=["dt_notific", "sg_uf", "evolucao"],
    filters=[("sg_uf", "==", "SP")],
)

print("shape:", df_sp.shape)
df_sp.head()